### **Semana 4 - Embeddings, segmentación y recuperación densa**

#### **Pregunta de la semana**

> En Semana 3 seleccionamos manualmente el contexto que entregábamos al LLM.  Luego, ¿cómo puede un sistema recuperar automáticamente el contexto más relevante para una consulta?.

```text
Semana 3
contexto seleccionado manualmente
        |
        v
Semana 4
documentos
    ->
fragmentos
    ->
embeddings
    ->
similitud
    ->
ordenamiento
    ->
top-k
```

El objetivo no es construir RAG ni aprender una API de almacenes vectoriales. El objetivo es entender el componente de **recuperación densa** y poder explicar qué hace cada etapa.

#### **1. Configuración y reproducibilidad**

La ejecución canónica usa un codificador entrenado para recuperación de información:

```text
intfloat/multilingual-e5-small
```

Para comprobar la estructura sin descargar el modelo:

```bash
CC0F4_RUN_REAL_RETRIEVAL=0
```

Ese modo utiliza TF-IDF + SVD solo para validar el software. Sus números **no** constituyen evidencia sobre recuperación densa con el modelo canónico.

El modelo produce vectores de 384 dimensiones y admite secuencias de hasta 512 tokens. Para recuperación se preserva el convenio requerido por el modelo:

```text
query: <consulta>
passage: <texto indexado>
```

Los prefijos `query:` y `passage:` se mantienen en inglés porque forman parte de la interfaz esperada por el modelo.

El límite de tokens importa: si una condición de segmentación provoca truncación, ya no estaríamos modificando solamente la granularidad.

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

SEED = 42
np.random.seed(SEED)

MODEL_ID = "intfloat/multilingual-e5-small"
RUN_REAL_RETRIEVAL = (
    os.environ.get("CC0F4_RUN_REAL_RETRIEVAL", "1") == "1"
)

print("MODEL_ID:", MODEL_ID)
print("RUN_REAL_RETRIEVAL:", RUN_REAL_RETRIEVAL)

#### **2. De representaciones dispersas a densas**

Una representación dispersa (`sparse`) describe un texto mediante dimensiones ligadas explícitamente al vocabulario:

```text
documento -> conteos/TF-IDF -> vector de alta dimensión y disperso
```

Un embedding denso cambia la interfaz:

```text
texto
    ->
codificador
    ->
vector denso en R^d
```

Este bloque introduce la transición desde representaciones dispersas hacia embeddings densos.

In [ ]:
toy_documents = [
    "la red inalámbrica presenta latencia alta",
    "el wifi del campus está congestionado",
    "el repositorio conserva versiones de una tesis",
    "una licencia concurrente no tiene asientos libres",
    "el kernel de jupyter usa otro entorno de python",
]

count_vectorizer = CountVectorizer()
sparse_matrix = count_vectorizer.fit_transform(toy_documents)

sparse_df = pd.DataFrame(
    sparse_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(len(toy_documents))],
)

sparse_df

La matriz anterior permite inspeccionar literalmente las palabras presentes.

```text
ventaja:
interpretación directa

limitación:
compartir significado sin compartir palabras no está garantizado
```

Semana 5 recuperará esta familia de representaciones dispersas mediante BM25. Hoy nos concentramos en embeddings densos.

#### **3. Similitud coseno**

Para dos vectores $x$ y $y$:

$$
\cos(x,y)=\frac{x^\top y}{\|x\|_2\|y\|_2}
$$

Si normalizamos ambos vectores:

$$
\hat{x}^\top\hat{y}=\cos(x,y)
$$

Por tanto, para vectores L2-normalizados:

```text
similitud coseno == producto interno
```

Esta equivalencia será importante para entender `FAISS IndexFlatIP`.

In [ ]:
def l2_normalize(vector: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vector)

    if norm == 0:
        raise ValueError("No se puede normalizar el vector cero.")

    return vector / norm


x = np.array([2.0, 1.0, 0.0])
y = np.array([1.0, 2.0, 0.0])

cosine_xy = float(
    np.dot(x, y)
    / (np.linalg.norm(x) * np.linalg.norm(y))
)

normalized_dot = float(
    np.dot(
        l2_normalize(x),
        l2_normalize(y),
    )
)

print("coseno:", round(cosine_xy, 6))
print("producto_interno(normalizados):", round(normalized_dot, 6))

assert np.isclose(cosine_xy, normalized_dot)

#### **4. Un codificador de texto real**

Un codificador transforma cada texto en un vector:

```text
texto_1 -> e_1
texto_2 -> e_2
...
texto_n -> e_n

E in R^(n x d)
```

El modelo permanecerá fijo durante todo el experimento de Semana 4.

In [ ]:
mini_corpus = [
    "Un error DNS puede impedir resolver nombres aunque exista conectividad IP.",
    "La VPN permite acceder remotamente a servicios internos de la universidad.",
    "Un conflicto de Git debe resolverse entendiendo los cambios de ambas ramas.",
    "Los checksums permiten detectar cambios accidentales en archivos de investigación.",
    "Un kernel de Jupyter puede apuntar a un intérprete Python distinto.",
    "Una licencia concurrente puede rechazar usuarios cuando todos los asientos están ocupados.",
    "Un embargo restringe temporalmente el acceso público al PDF de una tesis.",
    "El phishing intenta obtener credenciales mediante mensajes o enlaces engañosos.",
]

mini_queries = [
    "Tengo red, pero no puedo resolver nombres de dominio.",
    "¿Cómo detecto si un archivo cambió?",
    "Jupyter no encuentra un paquete que sí instalé.",
]

print("Documentos:", len(mini_corpus))
print("Consultas:", len(mini_queries))

In [ ]:
class OfflineDenseEncoder:
    # Representación para validar el cuaderno. No sustituye al codificador canónico.

    def __init__(self, seed: int = 42):
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            lowercase=True,
        )
        self.seed = seed

    def fit_transform(
        self,
        documents: list[str],
        queries: list[str],
    ) -> tuple[np.ndarray, np.ndarray]:
        doc_sparse = self.vectorizer.fit_transform(documents)
        query_sparse = self.vectorizer.transform(queries)

        n_components = min(
            32,
            doc_sparse.shape[0] - 1,
            doc_sparse.shape[1] - 1,
        )

        if n_components < 2:
            raise ValueError(
                "El corpus es demasiado pequeño para el modelo sustituto de validación."
            )

        svd = TruncatedSVD(
            n_components=n_components,
            random_state=self.seed,
        )

        doc_dense = svd.fit_transform(doc_sparse)
        query_dense = svd.transform(query_sparse)

        return (
            normalize(doc_dense, norm="l2").astype("float32"),
            normalize(query_dense, norm="l2").astype("float32"),
        )


if RUN_REAL_RETRIEVAL:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(MODEL_ID)

    document_inputs = [
        f"passage: {text}"
        for text in mini_corpus
    ]

    query_inputs = [
        f"query: {text}"
        for text in mini_queries
    ]

    document_token_lengths = [
        len(
            encoder.tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
        for text in document_inputs
    ]

    max_document_tokens = max(
        document_token_lengths
    )

    if max_document_tokens > encoder.max_seq_length:
        raise ValueError(
            "El corpus de demostración excede la ventana "
            f"del encoder: {max_document_tokens} > "
            f"{encoder.max_seq_length}."
        )

    doc_embeddings = encoder.encode(
        document_inputs,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    query_embeddings = encoder.encode(
        query_inputs,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
else:
    print(
        "MODO VALIDACION: TF-IDF + SVD. "
        "No usar estos resultados como evidencia del curso."
    )

    encoder = OfflineDenseEncoder(seed=SEED)

    doc_embeddings, query_embeddings = encoder.fit_transform(
        mini_corpus,
        mini_queries,
    )

print("doc_embeddings:", doc_embeddings.shape)
print("query_embeddings:", query_embeddings.shape)

assert doc_embeddings.ndim == 2
assert query_embeddings.ndim == 2
assert doc_embeddings.shape[1] == query_embeddings.shape[1]

#### **5. Búsqueda exacta con NumPy**

Con embeddings normalizados:

$$
S=Q E^\top
$$

$S_{ij}$ es el producto interno entre la consulta $i$ y el documento $j$.

La operación compara cada consulta con todos los vectores, por lo que es una búsqueda **exacta**.

In [ ]:
similarity_matrix = query_embeddings @ doc_embeddings.T


def exact_top_k(
    scores: np.ndarray,
    k: int,
) -> tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[1])

    indices = np.argsort(
        -scores,
        axis=1,
    )[:, :k]

    values = np.take_along_axis(
        scores,
        indices,
        axis=1,
    )

    return values, indices


exact_scores, exact_indices = exact_top_k(
    similarity_matrix,
    k=3,
)

rows = []

for q_idx, query in enumerate(mini_queries):
    for rank, (score, doc_idx) in enumerate(
        zip(
            exact_scores[q_idx],
            exact_indices[q_idx],
        ),
        start=1,
    ):
        rows.append(
            {
                "query": query,
                "rank": rank,
                "score": float(score),
                "document": mini_corpus[int(doc_idx)],
            }
        )

pd.DataFrame(rows)

#### **6. Segmentación**

En recuperación de información normalmente no indexamos un documento largo como una sola unidad.

```text
documento -> passages -> chunks -> embeddings
```

- fragmento pequeño: mayor granularidad y menor contexto,
- fragmento grande: mayor contexto y posible dilución de la señal,
- solapamiento: conserva continuidad, pero duplica información.

El laboratorio mantendrá las demás variables fijas y cambiará solo el tamaño objetivo del fragmento.

In [ ]:
demo_passages = [
    "La VPN permite acceder a servicios internos desde fuera del campus.",
    "El inicio de sesión utiliza autenticación multifactor.",
    "Al cambiar de teléfono puede ser necesario registrar nuevamente el segundo factor.",
    "Algunos nombres internos solo resuelven con los DNS instalados por la VPN.",
    "Los logs pueden contener direcciones IP y deben revisarse antes de compartirse.",
]


def group_passages(
    passages: list[str],
    target_words: int,
) -> list[str]:
    chunks = []
    current = []
    current_words = 0

    for passage in passages:
        n_words = len(passage.split())

        if (
            current
            and current_words + n_words > target_words
        ):
            chunks.append(" ".join(current))
            current = []
            current_words = 0

        current.append(passage)
        current_words += n_words

    if current:
        chunks.append(" ".join(current))

    return chunks


for target_words in [30, 60]:
    chunks = group_passages(
        demo_passages,
        target_words=target_words,
    )

    print()
    print("target_words =", target_words)
    print("n_chunks =", len(chunks))

    for idx, chunk in enumerate(chunks):
        print(
            f"chunk_{idx}: {len(chunk.split())} palabras | {chunk}"
        )

#### **7. FAISS `IndexFlatIP`**

El patrón mínimo reutilizado aquí es:

```python
index = faiss.IndexFlatIP(d)
index.add(vectors)
scores, indices = index.search(query_vectors, k)
```

Precisión conceptual:

```text
IndexFlatIP = búsqueda exhaustiva por producto interno = búsqueda exacta
```

HNSW, IVF y Product Quantization se mantienen en inglés porque son nombres técnicos de métodos o familias de índices. Quedan como temas de exposición o de semanas posteriores.

In [ ]:
class NumpyFlatIP:
    # Sustituto para validar la interfaz cuando FAISS no está disponible.

    def __init__(self, dimension: int):
        self.dimension = dimension
        self.vectors = None

    def add(self, vectors: np.ndarray) -> None:
        vectors = np.asarray(
            vectors,
            dtype="float32",
        )

        if (
            vectors.ndim != 2
            or vectors.shape[1] != self.dimension
        ):
            raise ValueError("Dimensión incompatible.")

        self.vectors = vectors

    def search(
        self,
        query_vectors: np.ndarray,
        k: int,
    ) -> tuple[np.ndarray, np.ndarray]:
        if self.vectors is None:
            raise RuntimeError("El índice está vacío.")

        scores = (
            np.asarray(query_vectors, dtype="float32")
            @ self.vectors.T
        )

        return exact_top_k(scores, k)


if RUN_REAL_RETRIEVAL:
    import faiss

    index = faiss.IndexFlatIP(
        doc_embeddings.shape[1]
    )
else:
    index = NumpyFlatIP(
        doc_embeddings.shape[1]
    )

index.add(
    doc_embeddings.astype("float32")
)

index_scores, index_indices = index.search(
    query_embeddings.astype("float32"),
    3,
)

print("Ordenamiento NumPy:")
print(exact_indices)
print()
print("Ordenamiento del índice:")
print(index_indices)

assert np.array_equal(
    exact_indices,
    index_indices,
)

assert np.allclose(
    exact_scores,
    index_scores,
    atol=1e-5,
)

#### **8. Índice, almacén vectorial y recuperador no son sinónimos**

```text
modelo de embeddings -> produce vectores

métrica de similitud -> define cercanía

índice -> organiza y busca vectores

almacén vectorial -> vectores + ids + metadatos + persistencia

recuperador -> política de recuperación y evidencia
```

Una API puede ocultar estas capas, pero conceptualmente siguen siendo distintas.

#### **9. Una métrica mínima: Recall@k**

Si conocemos la evidencia relevante $G_q$ para una consulta y los primeros $k$ fragmentos cubren evidencia $C_k(q)$:

$$
Recall@k(q)
=
\frac{|G_q \cap C_k(q)|}{|G_q|}
$$

En el laboratorio compararemos:

```text
target_words = 80
target_words = 180
target_words = 320
        ->
Recall@1
Recall@3
Recall@5
```

`target_words` se mantiene en inglés porque es el nombre de una variable del experimento.

La evaluación más amplia con MRR y nDCG queda para Semana 7.

#### **10. Cierre**

Debes poder defender:

```text
representación dispersa != representación densa

embedding != documento

coseno(vectores normalizados) == producto interno(vectores normalizados)

IndexFlatIP == búsqueda exacta

índice != almacén vectorial != recuperador

más fragmentos != mejor recuperación automáticamente

top-k mayor != mejor sistema automáticamente
```

#### **Pregunta**

> ¿Cómo cambia la recuperación de evidencia al modificar únicamente la granularidad de la segmentación?

### **Ejercicios adicionales**


#### **Ejercicio 1 - Coseno y producto interno**

Construye tres vectores no normalizados y calcule:

```text
similitud coseno
producto interno
```

Luego normaliza los vectores con `l2_normalize(...)` y vuelve a calcular ambas cantidades.

Responde:

1. ¿Son iguales coseno y producto interno antes de normalizar?
2. ¿Qué ocurre después de la normalización?
3. ¿Por qué esta propiedad permite usar `IndexFlatIP` para ordenar por similitud coseno cuando los embeddings están L2-normalizados?.

#### **Ejercicio 2 - Auditar una consulta**

Agrega una consulta nueva a `mini_queries` que no copie literalmente una oración de `mini_corpus`.

Ejemplo de estructura:

```python
nueva_consulta = "..."
```

Recupera los tres primeros resultados y presente:

```text
consulta
rank
score
documento
```

Explica por qué el primer resultado es o no es semánticamente adecuado.

No es suficiente afirmar que "el score es alto". Debes relacionar el resultado con el contenido del documento.


#### **Ejercicio 3 - Efecto de top-k**

Para una misma consulta compara:

```text
top-k = 1
top-k = 3
top-k = 5
```

Responde:

1. ¿Qué evidencia adicional aparece al aumentar `k`?
2. ¿Aparecen resultados menos relacionados?
3. ¿Por qué aumentar `top-k` puede mejorar cobertura pero también introducir ruido?.

No se utiliza todavía un LLM para responder la consulta.


#### **Ejercicio 4 - Granularidad de la segmentación**

Utiliza `group_passages(...)` con:

```text
target_words = 30
target_words = 60
target_words = 90
```

Para cada condición reporta:

```text
número de fragmentos
longitud media en palabras
longitud mínima
longitud máxima
```

Antes de ejecutar escribe una hipótesis:

> Al aumentar `target_words`, espero que...

Después de ejecutar indique si el resultado confirma o contradice tu hipótesis.

#### **Ejercicio 5 - NumPy vs FAISS**

Para las consultas del cuaderno compara el ordenamiento producido por:

```text
matriz NumPy
vs
FAISS IndexFlatIP
```

Verifica programáticamente:

```python
np.array_equal(exact_indices, index_indices)
```

Responde:

1. ¿Por qué esperamos el mismo ordenamiento?
2. ¿Qué tendría que cambiar para que la comparación fuera entre búsqueda exacta y búsqueda aproximada?
3. ¿Podemos usar este ejercicio para afirmar que FAISS siempre es más rápido? Justifica.


#### **Ejercicio 6 - Diseñar el experimento del laboratorio**

Antes de ejecutar el laboratorio del jueves completa:

```text
Pregunta:
Hipótesis:
Línea base:
Variable independiente:
Variables fijas:
Métrica principal:
Resultado esperado:
```

Usa estas condiciones:

```text
small    = 80
baseline = 180
large    = 320
```

y mantiene fijos:

```text
corpus
queries
qrels
overlap_passages
modelo de embeddings
normalización L2
métrica de similitud
IndexFlatIP
top-k
```

No cambies la hipótesis después de observar los resultados.



#### **Ejercicio 7 - Análisis de error**

Después de ejecutar el laboratorio selecciona una consulta donde una condición de segmentación recupere peor evidencia que otra.

Muestra:

```text
query_id
consulta
evidencia relevante
fragmentos recuperados
scores
Recall@3
```

Explica el error en términos de:

```text
granularidad
contexto incluido
información irrelevante
posición de la evidencia
```

Evita explicaciones genéricas como:

> El modelo se confundió.



#### **Ejercicio 8 - Conclusión técnica**

Escribe una conclusión de máximo ocho líneas que incluya:

```text
qué se comparó
qué variable cambió
qué variables permanecieron fijas
qué ocurrió con Recall@3
un caso de error
una limitación
qué NO puede generalizarse
```

Una conclusión adecuada debe comenzar con una formulación del tipo:

> Bajo este corpus, estas consultas, estos qrels y este modelo de embeddings, se observó...


## Tus respuestas